# Caso 8 — Modificação de Consultas

Selecionamos 5 consultas e produzimos, manualmente, uma versão alternativa
de cada uma, cobrindo diferentes estratégias de edição: remover termos,
acrescentar termos, usar sinônimos, tornar a consulta mais específica ou
mais genérica. Cada par (original, modificada) é executado nos dois
modelos (Modelo Vetorial e BM25) usando a mesma configuração de
pré-processamento (stopwords + stemming) do restante do projeto, e
comparamos o Top-10 antes/depois.


In [1]:
from pathlib import Path
import sys
project_root = Path.cwd()
if not (project_root / "src").is_dir() and (project_root.parent / "src").is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

import pandas as pd
from IPython.display import display

from src.cranfield_data import load_cranfield
from src.pre_processing import load_preprocessed, tokenize, apply_config, PREPROCESSING_CONFIGS
from src.vector_model import VectorSpaceModel
from src.bm25 import BM25

df_docs, df_queries, df_qrels = load_cranfield()
preprocessed = load_preprocessed(project_root / "data" / "processed" / "preprocessed_cranfield.pkl")

CONFIG = "stopwords_stemming"
CFG_PARAMS = PREPROCESSING_CONFIGS[CONFIG]
doc_tokens = preprocessed[CONFIG]["docs"]
query_tokens_list = preprocessed[CONFIG]["queries"]
doc_ids = df_docs["doc_id"].tolist()
query_ids = df_queries["query_id"].tolist()
doc_title = dict(zip(df_docs.doc_id, df_docs.title.str.replace("\n", " ")))
query_text = dict(zip(df_queries.query_id, df_queries.text.str.replace("\n", " ")))

vsm = VectorSpaceModel(doc_tokens)
bm25 = BM25(doc_tokens, k1=1.2, b=0.75)


def preprocess_text(raw_text):
    """Aplica a MESMA configuração de pré-processamento (stopwords + stemming)
    usada em toda a coleção a um texto de consulta arbitrário."""
    tokens = tokenize(raw_text)
    return apply_config(tokens, CFG_PARAMS["remove_stopwords"], CFG_PARAMS["apply_stemming"])

## As 5 consultas e suas versões modificadas

In [2]:
# As 5 consultas escolhidas e suas versões modificadas manualmente,
# cobrindo as estratégias sugeridas no enunciado: remoção de termos,
# acréscimo de termos, sinônimos, e tornar mais específica/genérica.
QUERY_MODIFICATIONS = [
    {
        "query_id": "5",
        "strategy": "Remoção de termo (mais genérica)",
        "modified_text": "what chemical kinetic system is applicable to aerodynamic problems .",
    },
    {
        "query_id": "40",
        "strategy": "Substituição por sinônimos",
        "modified_text": "how can one identify transition phenomena in high-speed wakes .",
    },
    {
        "query_id": "100",
        "strategy": "Acréscimo de termo (mais específica)",
        "modified_text": (
            "what are the effects of initial imperfections on the elastic "
            "buckling of thin-walled cylindrical shells under axial compression ."
        ),
    },
    {
        "query_id": "150",
        "strategy": "Remoção de termos (mais genérica)",
        "modified_text": "what is the magnitude of second-order wing-body interference .",
    },
    {
        "query_id": "180",
        "strategy": "Acréscimo de termo (mais específica)",
        "modified_text": "how does scale height vary with altitude in an isothermal atmosphere .",
    },
]

for mod in QUERY_MODIFICATIONS:
    qid = mod["query_id"]
    print(f"Consulta {qid} [{mod['strategy']}]")
    print(f"  original : {query_text[qid]}")
    print(f"  modificada: {mod['modified_text']}")
    print()

Consulta 5 [Remoção de termo (mais genérica)]
  original : what chemical kinetic system is applicable to hypersonic aerodynamic problems .
  modificada: what chemical kinetic system is applicable to aerodynamic problems .

Consulta 40 [Substituição por sinônimos]
  original : how can one detect transition phenomena in hypersonic wakes .
  modificada: how can one identify transition phenomena in high-speed wakes .

Consulta 100 [Acréscimo de termo (mais específica)]
  original : what are the effects of initial imperfections on the elastic buckling of cylindrical shells under axial compression .
  modificada: what are the effects of initial imperfections on the elastic buckling of thin-walled cylindrical shells under axial compression .

Consulta 150 [Remoção de termos (mais genérica)]
  original : what is the magnitude of second-order wing-body interference at high supersonic mach number .
  modificada: what is the magnitude of second-order wing-body interference .

Consulta 180 [Acrésc

## Comparação de rankings: original vs. modificada

In [3]:
def top_n_doc_ids(model, tokens, n=10):
    return [doc_ids[i] for i, _ in model.rank(tokens, top_n=n)]


def compare_query(mod, n=10):
    qid = mod["query_id"]
    original_tokens = query_tokens_list[query_ids.index(qid)]
    modified_tokens = preprocess_text(mod["modified_text"])

    print("=" * 100)
    print(f"Consulta {qid} — {mod['strategy']}")
    print(f"  Original  : {query_text[qid]!r}")
    print(f"    tokens  : {original_tokens}")
    print(f"  Modificada: {mod['modified_text']!r}")
    print(f"    tokens  : {modified_tokens}")

    grades = dict(zip(df_qrels[df_qrels.query_id == qid].doc_id,
                       df_qrels[df_qrels.query_id == qid].relevance))

    for nome, model in [("BM25", bm25), ("Modelo Vetorial", vsm)]:
        top_orig = top_n_doc_ids(model, original_tokens, n)
        top_mod = top_n_doc_ids(model, modified_tokens, n)
        overlap = len(set(top_orig) & set(top_mod))
        print(f"\n  -- {nome}: sobreposição do Top-{n} = {overlap}/{n} --")
        rows = []
        for rank in range(n):
            d_o = top_orig[rank] if rank < len(top_orig) else None
            d_m = top_mod[rank] if rank < len(top_mod) else None
            rows.append({
                "rank": rank + 1,
                "doc_id (original)": d_o,
                "grau (original)": grades.get(d_o, "-") if d_o else "-",
                "doc_id (modificada)": d_m,
                "grau (modificada)": grades.get(d_m, "-") if d_m else "-",
            })
        display(pd.DataFrame(rows).set_index("rank"))
    print()


for mod in QUERY_MODIFICATIONS:
    compare_query(mod)


# Recalcula as sobreposicoes num DataFrame e persiste, para que a Tabela 4 do
# relatorio tenha origem num arquivo e nao numa leitura manual das saidas.
overlap_rows = []
for mod in QUERY_MODIFICATIONS:
    qid = mod["query_id"]
    orig = query_tokens_list[query_ids.index(qid)]
    modif = preprocess_text(mod["modified_text"])
    linha = {"query_id": qid, "estrategia": mod["strategy"]}
    for nome, model in [("BM25", bm25), ("Modelo Vetorial", vsm)]:
        top_o = set(top_n_doc_ids(model, orig, 10))
        top_m = set(top_n_doc_ids(model, modif, 10))
        linha[f"overlap_{nome}"] = len(top_o & top_m)
    overlap_rows.append(linha)

df_overlap = pd.DataFrame(overlap_rows)
RESULTS_DIR = project_root / "data" / "processed"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
df_overlap.to_csv(RESULTS_DIR / "modificacao_consultas_overlap.csv", index=False)
display(df_overlap)

Consulta 5 — Remoção de termo (mais genérica)
  Original  : 'what chemical kinetic system is applicable to hypersonic aerodynamic problems .'
    tokens  : ['chemic', 'kinet', 'system', 'applic', 'hyperson', 'aerodynam', 'problem']
  Modificada: 'what chemical kinetic system is applicable to aerodynamic problems .'
    tokens  : ['chemic', 'kinet', 'system', 'applic', 'aerodynam', 'problem']

  -- BM25: sobreposição do Top-10 = 6/10 --


,doc_id (original),grau (original),doc_id (modificada),grau (modificada)
rank,,,,
1,103,-,103,-
2,552,1,1032,-
3,1032,-,552,1
4,401,3,943,-
5,1296,1,968,-
6,943,-,401,3
7,968,-,368,-
8,625,-,746,-
9,1379,-,650,-



  -- Modelo Vetorial: sobreposição do Top-10 = 5/10 --


,doc_id (original),grau (original),doc_id (modificada),grau (modificada)
rank,,,,
1,103,-,103,-
2,552,1,1032,-
3,1032,-,367,-
4,367,-,552,1
5,410,-,410,-
6,1379,-,368,-
7,1158,-,968,-
8,401,3,1061,-
9,1272,-,746,-



Consulta 40 — Substituição por sinônimos
  Original  : 'how can one detect transition phenomena in hypersonic wakes .'
    tokens  : ['one', 'detect', 'transit', 'phenomena', 'hyperson', 'wake']
  Modificada: 'how can one identify transition phenomena in high-speed wakes .'
    tokens  : ['one', 'identifi', 'transit', 'phenomena', 'high', 'speed', 'wake']

  -- BM25: sobreposição do Top-10 = 3/10 --


...


Consulta 180 — Acréscimo de termo (mais específica)
  Original  : 'how does scale height vary with altitude in an atmosphere .'
    tokens  : ['scale', 'height', 'vari', 'altitud', 'atmospher']
  Modificada: 'how does scale height vary with altitude in an isothermal atmosphere .'
    tokens  : ['scale', 'height', 'vari', 'altitud', 'isotherm', 'atmospher']

  -- BM25: sobreposição do Top-10 = 10/10 --


,doc_id (original),grau (original),doc_id (modificada),grau (modificada)
rank,,,,
1,548,-1,548,-1
2,616,4,616,4
3,622,2,622,2
4,617,4,617,4
5,1324,-,1324,-
6,719,-,719,-
7,882,-,882,-
8,1391,-,1391,-
9,1103,-,1103,-



  -- Modelo Vetorial: sobreposição do Top-10 = 9/10 --


,doc_id (original),grau (original),doc_id (modificada),grau (modificada)
rank,,,,
1,616,4,616,4
2,622,2,622,2
3,548,-1,548,-1
4,314,-,314,-
5,1103,-,1103,-
6,617,4,617,4
7,218,-,81,-
8,613,-,218,-
9,806,-,613,-


,query_id,estrategia,overlap_BM25,overlap_Modelo Vetorial
0,5,Remoção de termo (mais genérica),6,5
1,40,Substituição por sinônimos,3,8
2,100,Acréscimo de termo (mais específica),9,9
3,150,Remoção de termos (mais genérica),7,10
4,180,Acréscimo de termo (mais específica),10,9


## O que muda, e por quê

Resumo das sobreposições de Top 10 entre a consulta original e a modificada:

| Consulta | Estratégia | Overlap BM25 | Overlap Vetorial |
|---|---|---|---|
| 5   | remover termo, mais genérica      | 6/10  | 5/10  |
| 40  | sinônimos                          | 3/10  | 8/10  |
| 100 | acrescentar termo, mais específica | 9/10  | 9/10  |
| 150 | remover termos, mais genérica      | 7/10  | 10/10 |
| 180 | acrescentar termo raro             | 10/10 | 9/10  |

**Sinônimos causam a maior ruptura, e só no BM25** (consulta 40). Trocar
`detect` por `identify` e `hypersonic` por `high-speed` muda os tokens de
`detect` e `hyperson` para `identifi`, `high` e `speed` — do ponto de vista
dos modelos, palavras sem relação nenhuma com as originais, já que nem TF-IDF
nem BM25 têm qualquer noção de sinonímia. O BM25 troca 7 dos 10 primeiros
colocados (3/10), enquanto o Vetorial mantém 8/10.

A assimetria tem uma causa concreta. No BM25 o score é uma **soma** de
contribuições por termo: retirar `hyperson`, um termo de idf alto, remove uma
parcela grande do total e promove documentos que pontuavam bem nos termos
restantes. No Vetorial, o cosseno é normalizado pela norma L2 do vetor da
consulta **inteira**, de modo que a troca redistribui pesos em vez de subtrair
uma parcela, e os quatro termos que não mudaram (`one`, `transit`,
`phenomena`, `wake`) continuam determinando a direção do vetor.

**Acrescentar um termo mexe pouco, mas mexe** (consultas 100 e 180). Na 100,
`thin-walled` acrescenta os tokens `thin` e `wall` e troca 1 dos 10 colocados
nos dois modelos (9/10). Na 180, `isothermal` acrescenta `isotherm`, um termo
que aparece em pouquíssimos documentos da coleção: o Top 10 do BM25 não muda
(10/10) e o do Vetorial muda 1 posição (9/10). Um termo raro só reordena o
topo se os documentos que o contêm também competirem bem nos demais termos da
consulta — caso contrário, o bônus de idf não é suficiente para trazer um
documento novo ao Top 10.

**Remover termos tem efeito moderado e assimétrico** (consultas 5 e 150). Na
150, retirar quatro termos (`high`, `superson`, `mach`, `number`) não muda
**nada** no Vetorial (10/10) e troca 3 posições no BM25 (7/10) — de novo a
mesma mecânica de soma contra direção. Na 5, retirar `hyperson` afeta os dois
de forma parecida (6/10 e 5/10), porque nessa consulta curta o termo removido
carregava boa parte do sinal em ambos os modelos.

**Padrão geral.** As edições aditivas, que preservam todos os termos
originais, quase não mudam o ranking. As edições subtrativas ou de
substituição têm efeito visível, e é aí que os dois modelos mais divergem em
magnitude de reação: em 4 das 5 consultas o BM25 se mostrou mais sensível à
edição do que o Modelo Vetorial.